# Universal GPU Stable Diffusion Document Editing Pipeline
This notebook runs seamlessly across all GPU architectures and environments, **requiring no local project files (100% self-contained)**. It is ideal for running on cloud environments (like AMD AI Developer Cloud, Google Colab, Kaggle, etc.).

### How to use on a fresh cloud VM:
1. Run the **Step 1** dependency installer (detects ROCm vs CUDA vs DirectML automatically and installs GPU-enabled PaddlePaddle).
2. Run the **Step 2 & 3** cells to download and initialize the local Stable Diffusion Img2Img and Inpainting pipelines on your GPU.
3. **Upload your document file** (e.g., `pal.jpeg` or `lol.pdf`) to the root directory of this Jupyter environment using the file explorer sidebar on the left.
4. Run the **Step 4** pipeline code cells (defines all helper functions including GPU inpainting and PDF rendering).
5. Run the **Step 5** cell. Enter your file name live, and it will render the PDF page (if applicable), execute PaddleOCRVL layout analysis on the GPU, and display an annotated block ID map.
6. Specify replacements in the **Step 6** cell and render your final edited document with GPU inpainting background reconstruction.

## Step 1: Install Dependencies
Run the following cell to install the necessary libraries for layout parsing, OCR, PyQt5 rendering, and image generation.

In [ ]:
import sys
import subprocess

def install(package):
    print(f"Installing {package}...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", package])

# Upgrade pip first
subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "pip"])

# Ensure torch is installed first (if not pre-installed)
try:
    import torch
    print(f"PyTorch already installed: {torch.__version__}")
except ImportError:
    print("PyTorch not found. Installing standard PyTorch...")
    install("torch torchvision torchaudio")

# 1. Hardware and environment detection
import torch
has_gpu = False
gpu_backend = "cpu"
if torch.cuda.is_available():
    has_gpu = True
    version_str = torch.__version__.lower()
    if "rocm" in version_str or "hip" in version_str:
        gpu_backend = "rocm"
    else:
        gpu_backend = "cuda"

print(f"[Detection] Active GPU Backend: {gpu_backend.upper()}")
if gpu_backend == "rocm":
    print(f"[Detection] AMD ROCm PyTorch version detected: {torch.__version__}")

# Install GPU-enabled PaddlePaddle if GPU is available
try:
    import paddle
    if not paddle.device.is_compiled_with_cuda() and gpu_backend in ("rocm", "cuda"):
        raise ImportError("Need GPU version of Paddle")
    print("PaddlePaddle with GPU support is already installed.")
except ImportError:
    print("Installing GPU-enabled paddlepaddle...")
    try:
        if gpu_backend == "rocm":
            subprocess.check_call([sys.executable, "-m", "pip", "install", "paddlepaddle-gpu==3.0.0b1", "-i", "https://www.paddlepaddle.org.cn/packages/stable/rocm/"])
        else:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "paddlepaddle-gpu", "-i", "https://www.paddlepaddle.org.cn/packages/stable/conda/"])
    except Exception:
        try:
            install("paddlepaddle-gpu")
        except Exception as e:
            print(f"Warning: failed to install paddlepaddle-gpu: {e}. Falling back to CPU version...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", "paddlepaddle==3.2.0", "-i", "https://www.paddlepaddle.org.cn/packages/stable/cpu/"])

# 2. Compile dependencies list
dependencies = [
    "diffusers", "transformers", "accelerate", "safetensors", 
    "opencv-python", "Pillow", "numpy", "PyQt5", "PyMuPDF"
]

if gpu_backend == "cpu":
    if sys.platform.startswith("win"):
        print("[Installation] Windows detected without CUDA. Installing torch-directml for local AMD GPU support...")
        dependencies.insert(0, "torch-directml")
    else:
        print("[Installation] Linux detected. If running on AMD Cloud Portal, ensure PyTorch ROCm is pre-installed.")
else:
    print(f"[Installation] Native GPU acceleration active ({gpu_backend.upper()}). Skipping torch-directml.")

for dep in dependencies:
    try:
        install(dep)
    except Exception as e:
        print(f"Warning: failed to install {dep}: {e}")

# Install paddleocr[all] last
try:
    import paddleocr
    print("PaddleOCR already installed.")
except ImportError:
    print("Installing paddleocr[all]...")
    try:
        install("paddleocr[all]")
    except Exception:
        install("paddleocr")

print("\n[Success] All dependencies installed successfully!")

## Step 2: Initialize Stable Diffusion Pipelines on GPU
This cell dynamically selects the best GPU acceleration provider and loads both the Stable Diffusion Img2Img pipeline (for image blocks) and the Stable Diffusion Inpainting pipeline (for background reconstruction).

In [ ]:
import os
import sys
import torch
from diffusers import StableDiffusionImg2ImgPipeline, StableDiffusionInpaintPipeline

# 1. Set WSL 2 GPU search path if running under WSL 2
if sys.platform.startswith("linux"):
    wsl_lib_path = "/usr/lib/wsl/lib"
    if os.path.exists(wsl_lib_path):
        current_ld = os.environ.get("LD_LIBRARY_PATH", "")
        if wsl_lib_path not in current_ld:
            os.environ["LD_LIBRARY_PATH"] = f"{wsl_lib_path}:{current_ld}".strip(":")
            print(f"[WSL Setup] Added {wsl_lib_path} to LD_LIBRARY_PATH dynamically.")

# 2. Detect best hardware accelerator device
device_type = "cpu"
if torch.cuda.is_available():
    device = torch.device("cuda")
    version_str = torch.__version__.lower()
    if "rocm" in version_str or "hip" in version_str:
        device_type = "rocm"
        print(f"[Device Selection] Using AMD AI Cloud acceleration via native ROCm/HIP device: {device}")
    else:
        device_type = "cuda"
        print(f"[Device Selection] Using Cloud GPU acceleration via native CUDA: {torch.cuda.get_device_name(0)}")
else:
    try:
        import torch_directml
        dml_device = torch_directml.device()
        test_tensor = torch.tensor([1.0]).to(dml_device)
        device = dml_device
        device_type = "dml"
        print(f"[Device Selection] Using Local AMD GPU acceleration via DirectML device: {device}")
    except Exception as e:
        device = torch.device("cpu")
        device_type = "cpu"
        print(f"[Device Selection] GPU not available. Falling back to CPU: {device}")

# 3. Load the Stable Diffusion v1.5 Img2Img pipeline
model_id = "runwayml/stable-diffusion-v1-5"
print(f"Loading Img2Img model '{model_id}' onto device...")

if device_type in ("rocm", "cuda"):
    sd_pipeline = StableDiffusionImg2ImgPipeline.from_pretrained(
        model_id,
        torch_dtype=torch.float16,
        safety_checker=None
    )
else:
    sd_pipeline = StableDiffusionImg2ImgPipeline.from_pretrained(
        model_id,
        torch_dtype=torch.float32
)
sd_pipeline = sd_pipeline.to(device)

# 4. Load the Stable Diffusion Inpainting pipeline for background reconstruction
inpaint_model_id = "runwayml/stable-diffusion-inpainting"
print(f"Loading Inpaint model '{inpaint_model_id}' onto device...")
if device_type in ("rocm", "cuda"):
    sd_inpaint_pipeline = StableDiffusionInpaintPipeline.from_pretrained(
        inpaint_model_id,
        torch_dtype=torch.float16,
        safety_checker=None
    )
else:
    sd_inpaint_pipeline = StableDiffusionInpaintPipeline.from_pretrained(
        inpaint_model_id,
        torch_dtype=torch.float32
)
sd_inpaint_pipeline = sd_inpaint_pipeline.to(device)

print("Stable Diffusion pipelines loaded successfully!")

## Step 3: Local Image-to-Image Generation Function
This function crops the target block, runs it through the Stable Diffusion model, and applies a `strength` parameter to match original noise, lighting, and textures.

In [ ]:
from PIL import Image

def generate_similar_image_local(block_image, prompt, strength=0.45, guidance_scale=7.5, steps=30):
    """
    Generates a replacement image matching the style, background noise, and dimensions of the original block.
    """
    init_image = block_image.convert("RGB")
    orig_w, orig_h = init_image.size
    
    # Scale block dimensions to multiples of 8 for Stable Diffusion
    target_w = max(256, min(768, ((orig_w + 7) // 8) * 8))
    target_h = max(256, min(768, ((orig_h + 7) // 8) * 8))
    
    resized_init = init_image.resize((target_w, target_h), Image.Resampling.LANCZOS)
    
    print(f"Local Gen: '{prompt}' | Dimensions: {target_w}x{target_h} | Strength: {strength}")
    
    # Match autocast precision context
    is_gpu = device_type in ("rocm", "cuda")
    autocast_ctx = torch.autocast("cuda") if is_gpu else torch.no_grad()
    
    with autocast_ctx:
        output = sd_pipeline(
            prompt=prompt,
            image=resized_init,
            strength=strength,
            guidance_scale=guidance_scale,
            num_inference_steps=steps
        )
        
    generated_img = output.images[0]
    return generated_img.resize((orig_w, orig_h), Image.Resampling.LANCZOS)

## Step 4: Full Pipeline Helper Functions
Run this cell to define all the helper functions for layout parsing, OCR processing, bounding box alignment, PDF rendering, GPU background inpainting, and offscreen PyQt5 painting.

In [ ]:
import os
import sys
import cv2
import json
import copy
import urllib.request
from pathlib import Path
import numpy as np
from PIL import Image

DEFAULT_FONTS_DIR = Path("./fonts")
DEFAULT_FONTS_DIR.mkdir(exist_ok=True)

def merge_bboxes(bboxes):
    if not bboxes: return None
    bboxes = np.array(bboxes)
    return [
        int(np.min(bboxes[:, 0])),
        int(np.min(bboxes[:, 1])),
        int(np.max(bboxes[:, 2])),
        int(np.max(bboxes[:, 3]))
    ]

def load_ocr_data(json_path):
    with open(json_path, "r", encoding="utf-8") as f:
        return json.load(f)

def render_input_file(input_path, page_index=0, pdf_dpi=300):
    import fitz
    ext = Path(input_path).suffix.lower()
    if ext == ".pdf":
        doc = fitz.open(str(input_path))
        try:
            if page_index < 0 or page_index >= doc.page_count:
                raise ValueError(f"PDF page index {page_index} is out of range. Pages: {doc.page_count}")
            page = doc.load_page(page_index)
            zoom = pdf_dpi / 72
            pix = page.get_pixmap(matrix=fitz.Matrix(zoom, zoom), alpha=False)
            return Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
        finally:
            doc.close()
    with Image.open(input_path) as image:
        return image.convert("RGB")

def detect_dominant_language(data):
    if not data or "parsing_res_list" not in data: return "en"
    script_counts = {}
    for block in data.get("parsing_res_list", []):
        content = block.get("block_content")
        if not content: continue
        for ch in str(content):
            cp = ord(ch)
            if 0x0900 <= cp <= 0x097F: script_counts["hi"] = script_counts.get("hi", 0) + 1
            elif 0x0980 <= cp <= 0x09FF: script_counts["ben"] = script_counts.get("ben", 0) + 1
            elif 0x0A80 <= cp <= 0x0AFF: script_counts["gu"] = script_counts.get("gu", 0) + 1
            elif 0x0B80 <= cp <= 0x0BFF: script_counts["ta"] = script_counts.get("ta", 0) + 1
            elif 0x0C00 <= cp <= 0x0C7F: script_counts["te"] = script_counts.get("te", 0) + 1
            elif 0x0C80 <= cp <= 0x0CFF: script_counts["kn"] = script_counts.get("kn", 0) + 1
            elif 0x0D00 <= cp <= 0x0D7F: script_counts["mal"] = script_counts.get("mal", 0) + 1
            elif 0x0600 <= cp <= 0x06FF or 0x0750 <= cp <= 0x077F or 0x08A0 <= cp <= 0x08FF:
                script_counts["ar"] = script_counts.get("ar", 0) + 1
    if not script_counts: return "en"
    return max(script_counts, key=script_counts.get)

def create_text_ocr(lang="en"):
    from paddleocr import PaddleOCR
    import torch
    use_gpu = torch.cuda.is_available()
    option_sets = [
        {"use_doc_orientation_classify": False, "use_doc_unwarping": False, "use_textline_orientation": False, "lang": lang, "use_gpu": use_gpu},
        {"use_angle_cls": False, "lang": lang, "use_gpu": use_gpu},
        {"lang": lang, "use_gpu": use_gpu}
    ]
    for options in option_sets:
        try:
            return PaddleOCR(**options)
        except Exception:
            continue
    if lang != "en":
        return create_text_ocr(lang="en")
    return PaddleOCR(use_angle_cls=False, lang="en", use_gpu=use_gpu)

def result_mapping(result):
    if isinstance(result, dict): return result
    for attr in ("json", "res", "data"):
        value = getattr(result, attr, None)
        if isinstance(value, dict): return value
    for method in ("to_dict", "dict"):
        fn = getattr(result, method, None)
        if callable(fn):
            value = fn()
            if isinstance(value, dict): return value
    return None

def first_present(mapping, names):
    for name in names:
        value = mapping.get(name)
        if value is None: continue
        try:
            if len(value) == 0: continue
        except TypeError:
            pass
        return value
    return []

def bbox_from_points(points):
    arr = np.array(points, dtype=float)
    if arr.size == 4 and arr.ndim == 1:
        return [int(v) for v in arr.tolist()]
    arr = arr.reshape(-1, 2)
    return [
        int(np.min(arr[:, 0])),
        int(np.min(arr[:, 1])),
        int(np.max(arr[:, 0])),
        int(np.max(arr[:, 1])),
    ]

def extract_entries_from_mapping(mapping):
    if "res" in mapping and isinstance(mapping["res"], dict):
        mapping = mapping["res"]
    texts = first_present(mapping, ("rec_texts", "texts", "text", "labels"))
    boxes = first_present(mapping, ("rec_boxes", "dt_boxes", "boxes"))
    polys = first_present(mapping, ("rec_polys", "dt_polys", "polys", "points"))
    scores = first_present(mapping, ("rec_scores", "scores"))
    geometry = boxes if len(boxes) else polys
    if isinstance(texts, str): texts = [texts]
    if not isinstance(scores, (list, tuple, np.ndarray)): scores = [scores]
    entries = []
    for index, text in enumerate(texts):
        if index >= len(geometry): continue
        text = str(text).strip()
        if not text: continue
        score = float(scores[index]) if index < len(scores) else None
        entries.append({
            "text": text,
            "bbox": bbox_from_points(geometry[index]),
            "score": score,
        })
    return entries

def extract_entries_from_legacy_result(raw_result):
    entries = []
    def visit(node):
        if isinstance(node, dict):
            entries.extend(extract_entries_from_mapping(node))
            return
        if (
            isinstance(node, (list, tuple))
            and len(node) == 2
            and isinstance(node[1], (list, tuple))
            and len(node[1]) >= 1
        ):
            points = node[0]
            text = node[1][0]
            score = node[1][1] if len(node[1]) > 1 else None
            if isinstance(text, str) and text.strip():
                entries.append({
                    "text": text.strip(),
                    "bbox": bbox_from_points(points),
                    "score": float(score) if score is not None else None,
                })
                return
        if isinstance(node, (list, tuple)):
            for item in node:
                visit(item)
    visit(raw_result)
    return entries

def extract_text_ocr_entries(raw_result):
    entries = []
    for result in raw_result if isinstance(raw_result, list) else [raw_result]:
        mapping = result_mapping(result)
        if mapping is not None:
            entries.extend(extract_entries_from_mapping(mapping))
        else:
            entries.extend(extract_entries_from_legacy_result(result))
    if not entries:
        entries.extend(extract_entries_from_legacy_result(raw_result))
    deduped = []
    seen = set()
    for entry in entries:
        key = (entry["text"], tuple(entry["bbox"]))
        if key in seen: continue
        seen.add(key)
        deduped.append(entry)
    return sorted(deduped, key=lambda item: (item["bbox"][1], item["bbox"][0]))

def merge_same_line_entries(entries):
    if not entries: return []
    sorted_entries = sorted(entries, key=lambda e: e["bbox"][1])
    lines = []
    for entry in sorted_entries:
        bbox = entry["bbox"]
        h = bbox[3] - bbox[1]
        if h <= 0: continue
        matched_line_idx = None
        for idx, line in enumerate(lines):
            ly1, ly2 = line["y1"], line["y2"]
            y1, y2 = bbox[1], bbox[3]
            overlap = max(0, min(ly2, y2) - max(ly1, y1))
            min_h = min(ly2 - ly1, y2 - y1)
            if min_h > 0 and (overlap / min_h) >= 0.5:
                matched_line_idx = idx
                break
        if matched_line_idx is not None:
            lines[matched_line_idx]["entries"].append(entry)
            lines[matched_line_idx]["y1"] = min(lines[matched_line_idx]["y1"], bbox[1])
            lines[matched_line_idx]["y2"] = max(lines[matched_line_idx]["y2"], bbox[3])
        else:
            lines.append({"y1": bbox[1], "y2": bbox[3], "entries": [entry]})
    merged_entries = []
    for line in lines:
        line_entries = sorted(line["entries"], key=lambda e: e["bbox"][0])
        texts = [e["text"] for e in line_entries]
        merged_text = " ".join(texts)
        bboxes = [e["bbox"] for e in line_entries]
        merged_bbox = merge_bboxes(bboxes)
        scores = [e["score"] for e in line_entries if e.get("score") is not None]
        avg_score = sum(scores) / len(scores) if scores else 1.0
        merged_entries.append({"text": merged_text, "bbox": merged_bbox, "score": avg_score})
    return sorted(merged_entries, key=lambda e: (e["bbox"][1], e["bbox"][0]))

def split_multiline_blocks_with_text_ocr(data, source_image, output_dir, run_label):
    multiline_blocks = [
        block for block in data["parsing_res_list"]
        if block.get("block_content") and str(block.get("block_content")).strip()
        and str(block.get("block_label", "")).lower() not in (
            "table", "image", "header_image", "footer_image", "figure", "table_text", "seal", "chart"
        )
    ]
    if not multiline_blocks: return data
    split_data = copy.deepcopy(data)
    scale_x = source_image.width / data["width"]
    scale_y = source_image.height / data["height"]
    back_scale_x = data["width"] / source_image.width
    back_scale_y = data["height"] / source_image.height

    lang = detect_dominant_language(data)
    text_ocr = create_text_ocr(lang=lang)
    
    max_block_id = max([int(block.get("block_id", -1)) for block in split_data["parsing_res_list"]] or [-1])
    next_block_id = max_block_id + 1
    new_blocks = []
    multiline_ids = {block.get("block_id") for block in multiline_blocks}

    for block in split_data["parsing_res_list"]:
        block_id = block.get("block_id")
        if block_id not in multiline_ids:
            new_blocks.append(block)
            continue
        content = block.get("block_content")
        bbox = block.get("block_bbox")
        if not content or not bbox or len(bbox) != 4:
            new_blocks.append(block)
            continue
            
        x1 = max(0, int(bbox[0] * scale_x) - 24)
        y1 = max(0, int(bbox[1] * scale_y) - 6)
        x2 = min(source_image.width, int(bbox[2] * scale_x) + 24)
        y2 = min(source_image.height, int(bbox[3] * scale_y) + 6)
        crop = source_image.crop((x1, y1, x2, y2))
        
        try:
            temp_crop_path = output_dir / f"temp_crop_{block_id}.png"
            crop.save(temp_crop_path)
            raw_result = text_ocr.ocr(str(temp_crop_path), cls=False)
            temp_crop_path.unlink(missing_ok=True)
            
            entries = extract_text_ocr_entries(raw_result)
            entries = merge_same_line_entries(entries)
        except Exception:
            entries = []

        if len(entries) > 1:
            for entry in entries:
                cb = entry["bbox"]
                mapped_bbox = [
                    int((x1 + cb[0]) * back_scale_x),
                    int((y1 + cb[1]) * back_scale_y),
                    int((x1 + cb[2]) * back_scale_x),
                    int((y1 + cb[3]) * back_scale_y),
                ]
                new_blocks.append({
                    "block_label": block.get("block_label", "text"),
                    "block_content": entry["text"],
                    "block_bbox": mapped_bbox,
                    "block_id": next_block_id,
                    "group_id": next_block_id,
                    "global_block_id": next_block_id,
                    "global_group_id": next_block_id,
                    "source_multiline_block_id": block_id,
                    "ocr_score": entry.get("score") or 1.0
                })
                next_block_id += 1
        else:
            new_blocks.append(block)
            
    split_data["parsing_res_list"] = new_blocks
    return split_data

def draw_block_overlay(data, source_image, output_path):
    img_cv = cv2.cvtColor(np.array(source_image).copy(), cv2.COLOR_RGB2BGR)
    img_h, img_w = img_cv.shape[:2]
    scale_x = img_w / data["width"]
    scale_y = img_h / data["height"]
    
    for block in data["parsing_res_list"]:
        if "block_id" not in block: continue
        bid = block["block_id"]
        bbox = block["block_bbox"]
        
        x1, y1 = int(bbox[0] * scale_x), int(bbox[1] * scale_y)
        x2, y2 = int(bbox[2] * scale_x), int(bbox[3] * scale_y)
        
        cv2.rectangle(img_cv, (x1, y1), (x2, y2), (255, 0, 0), 2, lineType=cv2.LINE_AA)
        label = f"ID {bid}"
        (w, h), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.4, 1)
        cv2.rectangle(img_cv, (x1, y1 - h - 6), (x1 + w + 4, y1), (255, 0, 0), -1)
        cv2.putText(
            img_cv, label, (x1 + 2, y1 - 3), 
            cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255), 1, lineType=cv2.LINE_AA
        )
    cv2.imwrite(str(output_path), img_cv)

def get_font_script(text):
    for ch in text:
        cp = ord(ch)
        if 0x0900 <= cp <= 0x097F: return "devanagari"
        if 0x0980 <= cp <= 0x09FF: return "bengali"
        if 0x0A00 <= cp <= 0x0A7F: return "gurmukhi"
        if 0x0A80 <= cp <= 0x0AFF: return "gujarati"
        if 0x0B00 <= cp <= 0x0B7F: return "oriya"
        if 0x0B80 <= cp <= 0x0BFF: return "tamil"
        if 0x0C00 <= cp <= 0x0C7F: return "telugu"
        if 0x0C80 <= cp <= 0x0CFF: return "kannada"
        if 0x0D00 <= cp <= 0x0D7F: return "malayalam"
        if 0x0600 <= cp <= 0x06FF or 0x0750 <= cp <= 0x077F or 0x08A0 <= cp <= 0x08FF: return "arabic"
    return "latin"

def get_font_family(text, font_families, bold=False):
    script = get_font_script(text)
    style = "bold" if bold else "regular"
    family = font_families.get(f"{script}_{style}")
    if not family:
        family = font_families.get(f"{script}_regular")
    if not family and bold:
        family = font_families.get("latin_bold")
    if not family:
        family = font_families.get("latin_regular")
    if not family:
        family = font_families.get("default")
    return family

def is_bold_block(block_id):
    return block_id in {22, 39, 74, 31, 32, 33, 34, 20, 22}

def normalize_replacement_lines(value):
    if not isinstance(value, list):
        return [{"text": str(value)}]
    normalized = []
    for line in value:
        if isinstance(line, dict): normalized.append(line)
        else: normalized.append({"text": str(line)})
    return normalized

def check_and_download_fonts(fonts_dir):
    fonts_dir.mkdir(parents=True, exist_ok=True)
    font_urls = {
        "NotoSans-Regular.ttf": "https://github.com/googlefonts/noto-fonts/raw/main/hinted/ttf/NotoSans/NotoSans-Regular.ttf",
        "NotoSans-Bold.ttf": "https://github.com/googlefonts/noto-fonts/raw/main/hinted/ttf/NotoSans/NotoSans-Bold.ttf",
        "NotoSansDevanagari-Regular.ttf": "https://github.com/googlefonts/noto-fonts/raw/main/hinted/ttf/NotoSansDevanagari/NotoSansDevanagari-Regular.ttf",
        "NotoSansDevanagari-Bold.ttf": "https://github.com/googlefonts/noto-fonts/raw/main/hinted/ttf/NotoSansDevanagari/NotoSansDevanagari-Bold.ttf",
        "NotoSansTelugu-Regular.ttf": "https://github.com/googlefonts/noto-fonts/raw/main/hinted/ttf/NotoSansTelugu/NotoSansTelugu-Regular.ttf",
        "NotoSansTelugu-Bold.ttf": "https://github.com/googlefonts/noto-fonts/raw/main/hinted/ttf/NotoSansTelugu/NotoSansTelugu-Bold.ttf",
        "NotoSansMalayalam-Regular.ttf": "https://github.com/googlefonts/noto-fonts/raw/main/hinted/ttf/NotoSansMalayalam/NotoSansMalayalam-Regular.ttf",
        "NotoSansMalayalam-Bold.ttf": "https://github.com/googlefonts/noto-fonts/raw/main/hinted/ttf/NotoSansMalayalam/NotoSansMalayalam-Bold.ttf",
        "NotoNastaliqUrdu-Regular.ttf": "https://github.com/googlefonts/noto-fonts/raw/main/hinted/ttf/NotoNastaliqUrdu/NotoNastaliqUrdu-Regular.ttf",
        "NotoNastaliqUrdu-Bold.ttf": "https://github.com/googlefonts/noto-fonts/raw/main/hinted/ttf/NotoNastaliqUrdu/NotoNastaliqUrdu-Bold.ttf",
        "NotoSansBengali-Regular.ttf": "https://github.com/googlefonts/noto-fonts/raw/main/hinted/ttf/NotoSansBengali/NotoSansBengali-Regular.ttf",
        "NotoSansBengali-Bold.ttf": "https://github.com/googlefonts/noto-fonts/raw/main/hinted/ttf/NotoSansBengali/NotoSansBengali-Bold.ttf",
        "NotoSansGujarati-Regular.ttf": "https://github.com/googlefonts/noto-fonts/raw/main/hinted/ttf/NotoSansGujarati/NotoSansGujarati-Regular.ttf",
        "NotoSansGujarati-Bold.ttf": "https://github.com/googlefonts/noto-fonts/raw/main/hinted/ttf/NotoSansGujarati/NotoSansGujarati-Bold.ttf",
        "NotoSansKannada-Regular.ttf": "https://github.com/googlefonts/noto-fonts/raw/main/hinted/ttf/NotoSansKannada/NotoSansKannada-Regular.ttf",
        "NotoSansKannada-Bold.ttf": "https://github.com/googlefonts/noto-fonts/raw/main/hinted/ttf/NotoSansKannada/NotoSansKannada-Bold.ttf",
        "NotoSansTamil-Regular.ttf": "https://github.com/googlefonts/noto-fonts/raw/main/hinted/ttf/NotoSansTamil/NotoSansTamil-Regular.ttf",
        "NotoSansTamil-Bold.ttf": "https://github.com/googlefonts/noto-fonts/raw/main/hinted/ttf/NotoSansTamil/NotoSansTamil-Bold.ttf",
    }
    for font_name, url in font_urls.items():
        font_path = fonts_dir / font_name
        if not font_path.exists():
            print(f"[Fonts] Missing {font_name}. Downloading...")
            try:
                req = urllib.request.Request(
                    url,
                    headers={'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
                )
                with urllib.request.urlopen(req) as response, open(font_path, 'wb') as out_file:
                    out_file.write(response.read())
                print(f"[Fonts] Downloaded {font_name} successfully.")
            except Exception as e:
                print(f"[Fonts] Failed to download {font_name}: {e}")

def reconstruct_backgrounds_gpu(image_path, bboxes, device_type, sd_inpaint_pipeline):
    """
    Uses GPU-accelerated Stable Diffusion Inpainting to erase target areas
    and reconstruct the document background texture and noise.
    """
    if not bboxes: return Image.open(image_path).convert("RGB")
    init_image = Image.open(image_path).convert("RGB")
    w, h = init_image.size
    
    # Create black mask
    mask = Image.new("L", (w, h), 0)
    from PIL import ImageDraw
    draw = ImageDraw.Draw(mask)
    for bbox in bboxes:
        draw.rectangle(bbox, fill=255)
        
    # Stable diffusion dimension requirements (multiples of 8)
    target_w = ((w + 7) // 8) * 8
    target_h = ((h + 7) // 8) * 8
    resized_init = init_image.resize((target_w, target_h), Image.Resampling.LANCZOS)
    resized_mask = mask.resize((target_w, target_h), Image.Resampling.NEAREST)
    
    print(f"Running GPU Background Inpainting for {len(bboxes)} regions...")
    is_gpu = device_type in ("rocm", "cuda")
    autocast_ctx = torch.autocast("cuda") if is_gpu else torch.no_grad()
    
    prompt = "clean paper background, seamless document texture, blank paper, scan, high resolution"
    with autocast_ctx:
        output = sd_inpaint_pipeline(
            prompt=prompt,
            image=resized_init,
            mask_image=resized_mask,
            strength=0.75,
            guidance_scale=7.5,
            num_inference_steps=30
        )
    generated_img = output.images[0]
    return generated_img.resize((w, h), Image.Resampling.LANCZOS)

def draw_replacements_qt(image_path, ocr_data, replacements, output_png, fonts_dir):
    check_and_download_fonts(fonts_dir)
    os.environ["QT_QPA_FONTDIR"] = str(fonts_dir)
    os.environ.setdefault("QT_QPA_PLATFORM", "offscreen")
    from PyQt5.QtCore import Qt, QRect
    from PyQt5.QtGui import QColor, QFont, QFontDatabase, QImage, QPainter
    from PyQt5.QtWidgets import QApplication
    
    app = QApplication.instance() or QApplication(sys.argv)
    
    # 1. Identify text replacement bboxes to erase
    with Image.open(image_path) as img:
        img_w, img_h = img.size
    scale_x = img_w / ocr_data["width"]
    scale_y = img_h / ocr_data["height"]
    
    text_bboxes = []
    for block in ocr_data["parsing_res_list"]:
        block_id = block.get("block_id")
        if block_id is None or block_id not in replacements: continue
        replacement_val = replacements[block_id]
        
        is_image = False
        if isinstance(replacement_val, str) and (replacement_val.startswith("prompt:") or replacement_val.startswith("image:")):
            is_image = True
            
        if not is_image:
            bbox = block["block_bbox"]
            x1 = int(bbox[0] * scale_x)
            y1 = int(bbox[1] * scale_y)
            x2 = int(bbox[2] * scale_x)
            y2 = int(bbox[3] * scale_y)
            text_bboxes.append([x1, y1, x2, y2])
            
    # 2. Run GPU Inpainting to reconstruct background
    temp_background_path = Path("temp_reconstructed_bg.png")
    if text_bboxes:
        print(f"Reconstructing background for {len(text_bboxes)} text regions on GPU...")
        inpainted_bg = reconstruct_backgrounds_gpu(image_path, text_bboxes, device_type, sd_inpaint_pipeline)
        inpainted_bg.save(temp_background_path)
        active_image_path = temp_background_path
    else:
        active_image_path = Path(image_path)
        
    # 3. Load the background image in QImage
    image = QImage(str(active_image_path))
    
    painter = QPainter()
    painter.begin(image)
    try:
        painter.setRenderHint(QPainter.TextAntialiasing)
        painter.setRenderHint(QPainter.Antialiasing)
        painter.setPen(QColor(0, 0, 0))
        
        font_files = {
            "latin_regular": "NotoSans-Regular.ttf",
            "latin_bold": "NotoSans-Bold.ttf",
            "devanagari_regular": "NotoSansDevanagari-Regular.ttf",
            "devanagari_bold": "NotoSansDevanagari-Bold.ttf",
            "telugu_regular": "NotoSansTelugu-Regular.ttf",
            "telugu_bold": "NotoSansTelugu-Bold.ttf",
            "malayalam_regular": "NotoSansMalayalam-Regular.ttf",
            "malayalam_bold": "NotoSansMalayalam-Bold.ttf",
            "arabic_regular": "NotoNastaliqUrdu-Regular.ttf",
            "arabic_bold": "NotoNastaliqUrdu-Bold.ttf",
            "bengali_regular": "NotoSansBengali-Regular.ttf",
            "bengali_bold": "NotoSansBengali-Bold.ttf",
            "gujarati_regular": "NotoSansGujarati-Regular.ttf",
            "gujarati_bold": "NotoSansGujarati-Bold.ttf",
            "kannada_regular": "NotoSansKannada-Regular.ttf",
            "kannada_bold": "NotoSansKannada-Bold.ttf",
            "tamil_regular": "NotoSansTamil-Regular.ttf",
            "tamil_bold": "NotoSansTamil-Bold.ttf",
        }
        font_families = {"default": app.font().family()}
        for key, font_name in font_files.items():
            font_path = fonts_dir / font_name
            if font_path.exists():
                font_id = QFontDatabase.addApplicationFont(str(font_path))
                if font_id != -1:
                    families = QFontDatabase.applicationFontFamilies(font_id)
                    if families:
                        font_families[key] = families[0]

        for block in ocr_data["parsing_res_list"]:
            block_id = block.get("block_id")
            if block_id is None or block_id not in replacements: continue
            
            bbox = block["block_bbox"]
            x1, y1 = int(bbox[0] * scale_x), int(bbox[1] * scale_y)
            x2, y2 = int(bbox[2] * scale_x), int(bbox[3] * scale_y)
            box_w, box_h = x2 - x1, y2 - y1
            
            replacement_val = replacements[block_id]
            is_image = False
            img_filename = None
            
            if isinstance(replacement_val, str) and replacement_val.startswith("prompt:"):
                pil_source = Image.open(image_path)
                orig_crop = pil_source.crop((x1, y1, x2, y2))
                gen_prompt = replacement_val[len("prompt:"):]
                
                print(f"Generating block {block_id} locally on GPU...")
                generated_img = generate_similar_image_local(orig_crop, gen_prompt, strength=0.45)
                
                img_filename = f"local_gen_{block_id}.png"
                generated_img.save(img_filename)
                is_image = True
            elif isinstance(replacement_val, str) and replacement_val.startswith("image:"):
                img_filename = replacement_val[len("image:"):]
                is_image = True
                
            if is_image and img_filename:
                img_to_draw = QImage(img_filename)
                painter.drawImage(QRect(x1, y1, box_w, box_h), img_to_draw)
                if Path(img_filename).exists() and "local_gen_" in img_filename:
                    Path(img_filename).unlink(missing_ok=True)
                continue
                
            line_specs = normalize_replacement_lines(replacements[block_id])
            text = "\n".join([str(line.get("text", "")) for line in line_specs])
            bold = is_bold_block(block_id)
            font_family = get_font_family(text, font_families, bold=bold)
            font_size = max(int(box_h * 0.45), 8)
            font = QFont(font_family, font_size)
            font.setBold(bold)
            painter.setFont(font)
            painter.drawText(x1, y1, box_w, box_h, Qt.TextWordWrap, text)
    finally:
        painter.end()
        if temp_background_path.exists():
            try: temp_background_path.unlink()
            except Exception: pass
            
    image.save(str(output_png))

## Step 5: Load Uploaded Document & Perform Layout OCR on GPU
Make sure you have uploaded your document file (e.g., `pal.jpeg` or `lol.pdf`) to this cloud Jupyter workspace.

Run this cell. It will ask you to type in the name of your file. Then, it will dynamically render the PDF page (if applicable), execute layout OCR on the cloud GPU, and draw the Block ID guide overlay map.

In [ ]:
from IPython.display import display, Image as DisplayImage

# 1. Prompt user for filename interactively
print("Please type the name of the file you want to edit (e.g. pal.jpeg, lol.pdf):")
input_filename = input("Filename: ").strip()

input_image_path = Path(input_filename)
if not input_image_path.exists():
    raise FileNotFoundError(f"File '{input_filename}' not found in directory. Please upload it and run again.")

# 2. Render PDF if input is a PDF file
ext = input_image_path.suffix.lower()
if ext == ".pdf":
    print(f"'{input_filename}' is a PDF. Rendering page index 0 to temporary image...")
    source_img = render_input_file(input_image_path, page_index=0)
    active_image_path = Path("temp_rendered_page_0.png")
    source_img.save(active_image_path)
else:
    active_image_path = input_image_path
    source_img = Image.open(active_image_path)

# 3. Run layout analyzer from scratch using the GPU
print("[OCR] Running PaddleOCRVL layout parser on GPU...")
from paddleocr import PaddleOCRVL

# Track existing JSON files before running to dynamically detect output
before_jsons = {p.resolve() for p in Path(".").glob("*.json")}

layout_pipeline = PaddleOCRVL()
output = layout_pipeline.predict(input=str(active_image_path))
structured_output = layout_pipeline.restructure_pages(list(output))

if not structured_output:
    raise ValueError("OCR Layout analysis returned no results.")

# Save layout JSON to disk
structured_output[0].save_to_json(save_path="./")

# Identify the newly created JSON file
after_jsons = {p.resolve() for p in Path(".").glob("*.json")}
new_jsons = sorted(after_jsons - before_jsons, key=lambda p: p.stat().st_mtime, reverse=True)

if new_jsons:
    raw_ocr_json = new_jsons[0]
else:
    fallback_json = Path(active_image_path.name).with_suffix(".json")
    if fallback_json.exists():
        raw_ocr_json = fallback_json
    else:
        all_jsons = sorted(Path(".").glob("*.json"), key=lambda p: p.stat().st_mtime, reverse=True)
        if all_jsons:
            raw_ocr_json = all_jsons[0]
        else:
            raise FileNotFoundError("Could not find the JSON layout file generated by PaddleOCRVL.")
            
print(f"[OCR] Raw layout JSON loaded from: {raw_ocr_json}")

# 4. Load layout details & split multiline blocks
ocr_data = load_ocr_data(raw_ocr_json)
output_dir = Path("./output_test")
output_dir.mkdir(exist_ok=True)

split_data = split_multiline_blocks_with_text_ocr(ocr_data, source_img, output_dir, "split")

# 5. Draw layout map showing all detected Block IDs
overlay_png = output_dir / "block_layout_map.png"
draw_block_overlay(split_data, source_img, overlay_png)

# 6. Display inline cell output
print("===========================================================")
print("LAYOUT ANALYSIS COMPLETE! Available blocks listed below:")
print("===========================================================")
for block in split_data["parsing_res_list"]:
    bid = block.get("block_id")
    content = block.get("block_content")
    label = block.get("block_label")
    if content and str(content).strip():
        print(f"ID {bid:2d} | Label: {label:10s} | Content: {repr(content)}")
    elif label in ("image", "header_image", "figure"):
        print(f"ID {bid:2d} | Label: {label:10s} | [Image/Graphic Area]")

print("\nSee annotated layout map below to match IDs with coordinates:")
display(DisplayImage(str(overlay_png)))

## Step 6: Enter Replacements and Render Outputs Inline
Use the block listing and layout map above to write your replacements in the `replacements` dictionary. 

*   To replace text, enter the string value directly (e.g. `21: "FATHER'S NAME"`).
*   To replace/generate graphics using the local AMD GPU-accelerated Stable Diffusion, use the **`prompt:`** prefix (e.g. `10: "prompt: a portrait..."`).

Run the cell below to perform edits, perform GPU-accelerated background inpainting, and view the completed image directly in Jupyter.

In [ ]:
# Write your replacements here based on the IDs shown in Step 5
replacements = {
    20: "SHAIK KHAJA SAIF AZAM",
    21: "FATHER'S NAME",
    22: "GHOUSE BASHA SHAIK",
    
    # Replace/regenerate the profile picture block (ID 10) locally on GPU:
    10: "prompt: a realistic passport photo of a smiling professional woman, high detail, studio portrait background, noise"
}

# Output file path
final_output_png = output_dir / "final_document_edited.png"

# Perform rendering (and local SD image generation)
print("Starting rendering process...")
draw_replacements_qt(active_image_path, split_data, replacements, final_output_png, DEFAULT_FONTS_DIR)
print(f"Edits completed successfully! Document saved to {final_output_png}")

# Clean up temporary rendered PDF page image if it exists
temp_rendered = Path("temp_rendered_page_0.png")
if temp_rendered.exists():
    try: temp_rendered.unlink()
    except Exception: pass

# Display the final rendered document inline
display(DisplayImage(str(final_output_png)))